In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

project_root = Path.cwd().parent.resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src import GPSICModel
from src.utils import confusion_matrix_contemporaneous, create_connections_contemporaneous

np.random.seed(10)

In this tutorial, we show how to run the $GP_{SIC}$-based algorithm for identifying contemporaneous causal interactions. We can load the simulated data from ``./contemporaneous/5nonlinear_lag1_n250.csv``, which was simulated using the ``data/simulate_contemporaneous_data.py`` script.

In [2]:
contemp_df = pd.read_csv("./contemporaneous/5nonlinear_lag1_n250.csv", header=None).to_numpy()
num_samples = 250
nvar = contemp_df.shape[1]
lags = 1
mc_runs = 10

# Ground-truth connections, zero-indexed
actual_cnnx = [(4, 0, 1), (1, 3, 1), (1, 0, 1), (0, 1, 1), (0, 3, 1), (3, 4, 0), (2, 3, 0)]
cnnx_list = create_connections_contemporaneous(nvar, lags=1)
actual_neg_cnnx = [tup for tup in cnnx_list if tup not in actual_cnnx]

We can then create a GPSICModel object with ``contemporaneous=True`` and call the ``SIC_contemporaneous`` method. 

In [3]:
gp_all_results = []
for i in range(mc_runs):
    gp_data = contemp_df[i*num_samples:(i+1)*num_samples][:]
    gp_model = GPSICModel(gp_data, num_samples, lags, contemporaneous=True) 
    gc_matrix = gp_model.SIC_contemporaneous()
    gp_all_results.append(gc_matrix.flatten())
# Create gp results dataframe
gp_df = pd.DataFrame(np.array(gp_all_results).astype(int), columns=cnnx_list)
gp_df = confusion_matrix_contemporaneous(gp_df, actual_cnnx, actual_neg_cnnx, cnnx_list)

In [4]:
gp_df

,"(0, 0, 0)","(0, 0, 1)","(0, 1, 0)","(0, 1, 1)","(0, 2, 0)","(0, 2, 1)","(0, 3, 0)","(0, 3, 1)","(0, 4, 0)","(0, 4, 1)",...,F1,Balanced Accuracy,Specificity_CAdj,Precision_CAdj,Recall_CAdj,F1_CAdj,Balanced Accuracy_CAdj,Precision_COr,Recall_COr,F1_COr
0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.0,...,0.75,0.8,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1,0.0,0.0,0.000000,1.000000,0.000000,0.0,0.000000,0.0,0.000000,0.0,...,0.888889,0.9,0.875,0.666667,1.0,0.8,0.9375,0.666667,1.0,0.8
2,0.0,0.0,0.000000,1.000000,0.000000,0.0,1.000000,0.0,0.000000,0.0,...,0.888889,0.9,0.875,0.666667,1.0,0.8,0.9375,0.666667,1.0,0.8
3,0.0,0.0,0.000000,1.000000,0.000000,0.0,0.000000,0.0,0.000000,0.0,...,0.888889,0.9,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
4,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.0,...,0.75,0.8,0.875,0.666667,1.0,0.8,0.9375,0.666667,1.0,0.8
5,0.0,0.0,1.000000,0.000000,1.000000,0.0,0.000000,0.0,0.000000,0.0,...,0.666667,0.775,0.75,0.5,1.0,0.666667,0.875,0.5,1.0,0.666667
6,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.0,...,0.75,0.8,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
7,0.0,0.0,0.000000,1.000000,0.000000,0.0,0.000000,0.0,1.000000,0.0,...,0.727273,0.85,0.875,0.666667,1.0,0.8,0.9375,0.333333,0.5,0.4
8,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.0,...,0.666667,0.775,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
9,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.0,...,0.75,0.8,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
